Домашнее задание: Прогнозирование зарплаты разработчика на основе данных опроса Stack Overflow
Цель работы: Применить нейросети для решения реальной задачи — предсказания зарплаты (или её аналога, например, ConvertedCompYearly) на основе ответов разработчиков на ежегодный опрос. Вы научитесь обрабатывать сложные смешанные данные (числовые, категориальные, множественного выбора), строить нейросетевые модели и оценивать их качество.

Источник данных: Stack Overflow Annual Developer Survey. Используйте данные последнего доступного года (например, 2024). Файл с результатами обычно доступен для скачивания в формате CSV.

# Блок 0: Постановка задачи
В этом задании вам предстоит построить нейросеть, которая по ответам респондента (его профессиональный опыт, используемые технологии, образование, страна проживания и т.д.) будет предсказывать его годовую зарплату. Это задача регрессии.

Целевая переменная: ConvertedCompYearly (годовая зарплата в долларах США).

Основные шаги:

Загрузить и изучить данные.

Провести предобработку: отбор признаков, обработка пропусков, кодирование.

Построить и обучить нейросеть в PyTorch.

Оценить качество модели.

# Блок 1: Загрузка данных

Перейдите на сайт Stack Overflow Developer Survey. Найдите раздел с данными последнего опроса (например, «Download the Data»). Скачайте файл с результатами (обычно это архив .zip с файлом .csv). Распакуйте архив и загрузите данные в pandas.

Ваша задача: Написать код для загрузки данных. Укажите правильный путь к файлу.

In [410]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

In [411]:
data = pd.read_csv('data/survey_results_public.csv')

In [412]:
data.describe()

,ResponseId,WorkExp,YearsCode,TechEndorse_1,TechEndorse_2,TechEndorse_3,TechEndorse_4,TechEndorse_5,TechEndorse_6,TechEndorse_7,...,SO_Actions_3,SO_Actions_4,SO_Actions_5,SO_Actions_6,SO_Actions_9,SO_Actions_7,SO_Actions_10,SO_Actions_15,ConvertedCompYearly,JobSat
count,49191.000000,42893.000000,43042.000000,35975.000000,35975.000000,35975.000000,35975.000000,35975.000000,35975.000000,35975.000000,...,26260.000000,26260.000000,26260.000000,26260.000000,26260.000000,26260.000000,26260.000000,26260.000000,2.394700e+04,26670.000000
mean,24596.000000,13.367403,16.570861,7.867352,4.104211,4.110271,5.678193,4.119388,5.225990,6.477387,...,5.718355,4.561767,4.790861,5.199657,5.676314,4.984653,7.099505,10.079284,1.017615e+05,7.201950
std,14200.362883,10.800117,11.787610,2.397432,2.275821,2.329536,2.398084,2.437945,2.801045,2.331468,...,2.628016,3.070548,2.643177,2.563562,2.310659,2.490095,2.469394,1.940928,4.617569e+05,1.997245
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000e+00,0.000000
25%,12298.500000,5.000000,8.000000,7.000000,2.000000,2.000000,4.000000,2.000000,3.000000,5.000000,...,3.000000,1.000000,3.000000,3.000000,4.000000,3.000000,6.000000,10.000000,3.817100e+04,6.000000
50%,24596.000000,10.000000,14.000000,9.000000,4.000000,4.000000,6.000000,4.000000,5.000000,7.000000,...,6.000000,4.000000,5.000000,5.000000,6.000000,5.000000,8.000000,10.000000,7.532000e+04,8.000000
75%,36893.500000,20.000000,24.000000,9.000000,6.000000,6.000000,8.000000,6.000000,8.000000,8.000000,...,8.000000,7.000000,7.000000,7.000000,7.000000,7.000000,9.000000,10.000000,1.205960e+05,8.000000
max,49191.000000,100.000000,100.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,...,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,5.000000e+07,10.000000


In [413]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Columns: 172 entries, ResponseId to JobSat
dtypes: float64(52), int64(1), str(119)
memory usage: 64.6 MB


In [414]:
data.head().T

,0,1,2,3,4
ResponseId,1,2,3,4,5
MainBranch,I am a developer by profession,I am a developer by profession,I am a developer by profession,I am a developer by profession,I am a developer by profession
Age,25-34 years old,25-34 years old,35-44 years old,35-44 years old,35-44 years old
EdLevel,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Associate degree (A.A., A.S., etc.)","Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Master’s degree (M.A., M.S., M.Eng., MBA, etc.)"
Employment,Employed,Employed,"Independent contractor, freelancer, or self-em...",Employed,"Independent contractor, freelancer, or self-em..."
...,...,...,...,...,...
AIAgentExtWrite,NaN,NaN,NaN,NaN,NaN
AIHuman,When I don’t trust AI’s answers,When I don’t trust AI’s answers;When I want to...,When I don’t trust AI’s answers;When I want to...,When I don’t trust AI’s answers;When I want to...,When I don’t trust AI’s answers
AIOpen,"Troubleshooting, profiling, debugging",All skills. AI is a flop.,"Understand how things actually work, problem s...",NaN,"critical thinking, the skill to define the tas..."
ConvertedCompYearly,61256.0,104413.0,53061.0,36197.0,60000.0


In [415]:
data.shape

(49191, 172)

In [416]:
data.columns.tolist()

['ResponseId',
 'MainBranch',
 'Age',
 'EdLevel',
 'Employment',
 'EmploymentAddl',
 'WorkExp',
 'LearnCodeChoose',
 'LearnCode',
 'LearnCodeAI',
 'AILearnHow',
 'YearsCode',
 'DevType',
 'OrgSize',
 'ICorPM',
 'RemoteWork',
 'PurchaseInfluence',
 'TechEndorseIntro',
 'TechEndorse_1',
 'TechEndorse_2',
 'TechEndorse_3',
 'TechEndorse_4',
 'TechEndorse_5',
 'TechEndorse_6',
 'TechEndorse_7',
 'TechEndorse_8',
 'TechEndorse_9',
 'TechEndorse_13',
 'TechEndorse_13_TEXT',
 'TechOppose_1',
 'TechOppose_2',
 'TechOppose_3',
 'TechOppose_5',
 'TechOppose_7',
 'TechOppose_9',
 'TechOppose_11',
 'TechOppose_13',
 'TechOppose_16',
 'TechOppose_15',
 'TechOppose_15_TEXT',
 'Industry',
 'JobSatPoints_1',
 'JobSatPoints_2',
 'JobSatPoints_3',
 'JobSatPoints_4',
 'JobSatPoints_5',
 'JobSatPoints_6',
 'JobSatPoints_7',
 'JobSatPoints_8',
 'JobSatPoints_9',
 'JobSatPoints_10',
 'JobSatPoints_11',
 'JobSatPoints_13',
 'JobSatPoints_14',
 'JobSatPoints_15',
 'JobSatPoints_16',
 'JobSatPoints_15_TEXT',
 

# Блок 3: Предобработка данных
Данные опроса содержат множество столбцов разного типа. Для построения модели мы выберем ограниченный набор признаков. Вам нужно обработать пропуски, закодировать категориальные переменные и подготовить данные для нейросети.

In [417]:
missing = data.isnull().sum()
missing.sort_values(ascending=False)

AIAgentObsWrite      48927
SOTagsWant Entry     48761
SOTagsHaveEntry      48733
AIModelsWantEntry    48716
AIAgentOrchWrite     48713
                     ...  
EdLevel               1042
Employment             852
ResponseId               0
MainBranch               0
Age                      0
Length: 172, dtype: int64

In [418]:
numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns
print(numeric_cols)
print(f'{len(numeric_cols)} - числовые признаки')

Index(['ResponseId', 'WorkExp', 'YearsCode', 'TechEndorse_1', 'TechEndorse_2',
       'TechEndorse_3', 'TechEndorse_4', 'TechEndorse_5', 'TechEndorse_6',
       'TechEndorse_7', 'TechEndorse_8', 'TechEndorse_9', 'TechEndorse_13',
       'TechOppose_1', 'TechOppose_2', 'TechOppose_3', 'TechOppose_5',
       'TechOppose_7', 'TechOppose_9', 'TechOppose_11', 'TechOppose_13',
       'TechOppose_16', 'TechOppose_15', 'JobSatPoints_1', 'JobSatPoints_2',
       'JobSatPoints_3', 'JobSatPoints_4', 'JobSatPoints_5', 'JobSatPoints_6',
       'JobSatPoints_7', 'JobSatPoints_8', 'JobSatPoints_9', 'JobSatPoints_10',
       'JobSatPoints_11', 'JobSatPoints_13', 'JobSatPoints_14',
       'JobSatPoints_15', 'JobSatPoints_16', 'ToolCountWork',
       'ToolCountPersonal', 'CompTotal', 'SO_Actions_1', 'SO_Actions_16',
       'SO_Actions_3', 'SO_Actions_4', 'SO_Actions_5', 'SO_Actions_6',
       'SO_Actions_9', 'SO_Actions_7', 'SO_Actions_10', 'SO_Actions_15',
       'ConvertedCompYearly', 'JobSat'],
     

In [419]:
category_cols = data.select_dtypes(include=['object']).columns
print(category_cols)
print(f'{len(category_cols)} - категориальные признаки')

Index(['MainBranch', 'Age', 'EdLevel', 'Employment', 'EmploymentAddl',
       'LearnCodeChoose', 'LearnCode', 'LearnCodeAI', 'AILearnHow', 'DevType',
       ...
       'AIAgentKnowledge', 'AIAgentKnowWrite', 'AIAgentOrchestration',
       'AIAgentOrchWrite', 'AIAgentObserveSecure', 'AIAgentObsWrite',
       'AIAgentExternal', 'AIAgentExtWrite', 'AIHuman', 'AIOpen'],
      dtype='str', length=119)
119 - категориальные признаки


In [420]:
data = data[data['ConvertedCompYearly'].notna()].copy()

In [421]:
data['ConvertedCompYearly'].isnull().sum()

np.int64(0)

In [422]:
# Выбор подходящих признаков для обучения модели + определение таргета
target = 'ConvertedCompYearly'
features = [
    'MainBranch',
    'Age',
    'EdLevel',
    'Employment',
    'WorkExp',
    'YearsCode',
    'DevType',
    'OrgSize',
    'Country',
]
# Создание новой таблицы
data_clean = data[features + [target]].copy()

In [423]:
# обработка пропусков у числовых признаков
data_clean['WorkExp'] = data_clean['WorkExp'].fillna(data_clean['WorkExp'].median())
data_clean['YearsCode'] = data_clean['YearsCode'].fillna(data_clean['YearsCode'].median())

# обработка пропусков у категориальных признаков
categorical_features = ['MainBranch','Age','EdLevel','Employment','DevType','OrgSize','Country']
for col in categorical_features:
    data_clean[col] = data_clean[col].fillna(data_clean[col].mode()[0])

data_clean.info()

<class 'pandas.DataFrame'>
Index: 23947 entries, 0 to 49122
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   MainBranch           23947 non-null  str    
 1   Age                  23947 non-null  str    
 2   EdLevel              23947 non-null  str    
 3   Employment           23947 non-null  str    
 4   WorkExp              23947 non-null  float64
 5   YearsCode            23947 non-null  float64
 6   DevType              23947 non-null  str    
 7   OrgSize              23947 non-null  str    
 8   Country              23947 non-null  str    
 9   ConvertedCompYearly  23947 non-null  float64
dtypes: float64(3), str(7)
memory usage: 2.0 MB


In [424]:
# Осмотр категориальных признаков
for col in categorical_features:
    print(data_clean[col].value_counts())
    print('-'*100)

MainBranch
I am a developer by profession                                                                20195
I am not primarily a developer, but I write code sometimes as part of my work/studies          2187
I used to be a developer by profession, but no longer am                                        498
I work with developers or my work supports developers but am not a developer by profession      390
I am learning to code                                                                           363
I code primarily as a hobby                                                                     314
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------
Age
25-34 years old      8598
35-44 years old      7582
45-54 years old      3429
18-24 years old      2601
55-64 years old      1387
65 years or older     328
Prefer not to say      22
Name: count, dtype: int64
--------------------------------------------------

In [425]:
# обработка пустых признаков
data_clean = data_clean[data_clean['Age'] != 'Prefer not to say']
data_clean = data_clean[data_clean['Employment'] != 'I prefer not to say']
data_clean = data_clean[data_clean['OrgSize'] != 'I don’t know']
data_clean = data_clean[data_clean['EdLevel'] != 'Other (please specify):']

In [426]:
# обработка MainBranch
strings_professional = [
    'I am a developer by profession',
    'I am not primarily a developer, but I write code sometimes as part of my work/studies',
    'I used to be a developer by profession, but no longer am'
]
data_clean['is_professional'] = data_clean['MainBranch'].isin(strings_professional).astype(int)
data_clean['is_professional'].value_counts()
data_clean.drop('MainBranch', axis=1, inplace=True)

In [427]:
# обработка Employed
strings_employed = [
    'Employed',
    'Independent contractor, freelancer, or self-employed'
]
data_clean['is_employed'] = data_clean['Employment'].isin(strings_employed).astype(int)
data_clean.drop('Employment', axis=1, inplace=True)

In [428]:
data_clean['Country'].value_counts()

Country
United States of America                                5107
Germany                                                 2072
United Kingdom of Great Britain and Northern Ireland    1452
India                                                   1063
France                                                  1006
                                                        ... 
Djibouti                                                   1
Turkmenistan                                               1
Burundi                                                    1
Belize                                                     1
Guinea                                                     1
Name: count, Length: 163, dtype: int64

In [429]:
# обработка Country
country_counts = data_clean['Country'].value_counts()
top_countries = country_counts.head(20).index.tolist()
data_clean['Country'] = data_clean['Country'].apply(
    lambda x: x if x in top_countries else 'Other'
)

In [430]:
y = data_clean['ConvertedCompYearly']
data_clean  = data_clean.drop('ConvertedCompYearly', axis=1)

In [431]:
X_train, X_test, y_train, y_test = train_test_split(
    data_clean, y, test_size=0.2, random_state=42
)

In [432]:
# Осмотр категориальных признаков
categorical_features = ['Age','EdLevel','DevType','OrgSize','Country']
for col in categorical_features:
    print(data_clean[col].value_counts())
    print('-'*100)

Age
25-34 years old      8392
35-44 years old      7431
45-54 years old      3357
18-24 years old      2493
55-64 years old      1356
65 years or older     319
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------
EdLevel
Bachelor’s degree (B.A., B.S., B.Eng., etc.)                                          10275
Master’s degree (M.A., M.S., M.Eng., MBA, etc.)                                        6821
Some college/university study without earning a degree                                 2827
Professional degree (JD, MD, Ph.D, Ed.D, etc.)                                         1325
Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)     1193
Associate degree (A.A., A.S., etc.)                                                     784
Primary/elementary school                                                               123
Name: count, dtype: int64
-----------------------------------

In [433]:
# Кодировка Age
age_order = [
    '18-24 years old',
    '25-34 years old',
    '35-44 years old',
    '45-54 years old',
    '55-64 years old',
    '65 years or older'
]

ord_enc = OrdinalEncoder(categories=[age_order])
X_train['Age'] = ord_enc.fit_transform(X_train[['Age']])
X_test['Age'] = ord_enc.transform(X_test[['Age']])

In [434]:
# Кодировка EdLevel
education_order = [
    'Primary/elementary school',
    'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)',
    'Some college/university study without earning a degree',
    'Associate degree (A.A., A.S., etc.)',
    'Bachelor’s degree (B.A., B.S., B.Eng., etc.)',
    'Master’s degree (M.A., M.S., M.Eng., MBA, etc.)',
    'Professional degree (JD, MD, Ph.D, Ed.D, etc.)'
]

ed_enc = OrdinalEncoder(categories=[education_order])
X_train['EdLevel'] = ed_enc.fit_transform(X_train[['EdLevel']])
X_test['EdLevel'] = ed_enc.transform(X_test[['EdLevel']])

In [435]:
X_train['DevType'].value_counts()

DevType
Developer, full-stack                            5786
Developer, back-end                              3102
Architect, software or solutions                 1278
Developer, desktop or enterprise applications     889
Developer, front-end                              886
Other (please specify):                           611
Developer, embedded applications or devices       610
Developer, mobile                                 606
Engineering manager                               574
DevOps engineer or professional                   537
Academic researcher                               427
Data engineer                                     393
Student                                           302
AI/ML engineer                                    279
Data scientist                                    273
Senior executive (C-suite, VP, etc.)              262
Cloud infrastructure engineer                     203
System administrator                              182
Founder, technology 

In [436]:
# Обработка 'DevType'
def group_roles(role):
    if pd.isna(role):
        return 'Other'
    role_lower = role.lower()
    if any(keyword in role_lower for keyword in ['data', 'ai', 'ml', 'scientist', 'analyst']):
        return 'Data & AI'
    if any(keyword in role_lower for keyword in ['architect', 'manager', 'executive', 'founder', 'lead']):
        return 'Architecture & Management'
    if any(keyword in role_lower for keyword in ['devops', 'cloud', 'infrastructure', 'administrator', 'admin', 'support']):
        return 'Infrastructure & DevOps'
    if any(keyword in role_lower for keyword in ['security', 'infosec', 'cyber']):
        return 'Security'
    if any(keyword in role_lower for keyword in ['ux', 'ui', 'design']):
        return 'Design'
    if any(keyword in role_lower for keyword in ['research', 'student']):
        return 'Research & Education'
    if 'developer' in role_lower or 'engineer' in role_lower:
        return 'Software Development'
    return 'Other'

X_train['DevType'] = X_train['DevType'].apply(group_roles)
X_test['DevType'] = X_test['DevType'].apply(group_roles)


In [437]:
X_train['DevType'].value_counts()

DevType
Software Development         12172
Architecture & Management     2478
Data & AI                     1535
Infrastructure & DevOps        922
Research & Education           729
Other                          666
Security                       141
Design                          35
Name: count, dtype: int64

In [438]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_train = ohe.fit_transform(X_train[['DevType']])
encoded_test = ohe.transform(X_test[['DevType']])

feature_names = ohe.get_feature_names_out(['DevType'])

train_devtype_df = pd.DataFrame(encoded_train, columns=feature_names, index=X_train.index)
test_devtype_df = pd.DataFrame(encoded_test, columns=feature_names, index=X_test.index)

X_train = pd.concat([X_train.drop('DevType', axis=1), train_devtype_df], axis=1)
X_test = pd.concat([X_test.drop('DevType', axis=1), test_devtype_df], axis=1)

In [439]:
X_train['OrgSize'].value_counts()

OrgSize
20 to 99 employees                                    5497
100 to 499 employees                                  3189
Less than 20 employees                                2651
10,000 or more employees                              2541
1,000 to 4,999 employees                              2182
500 to 999 employees                                  1238
5,000 to 9,999 employees                               831
Just me - I am a freelancer, sole proprietor, etc.     549
Name: count, dtype: int64

In [440]:
import re
def group_org_size(company_size):
    if 'Just me' in company_size:
        return 'Freelancer'
    clean_str = str(company_size).replace(',', '')
    match = re.search(r'\d+', clean_str)
    if match:
        num = int(match.group())
        if num < 20:
            return 'Micro (1-19)'
        elif num < 100:
            return 'Small (20-99)'
        elif num < 500:
            return 'Medium-Small (100-499)'
        elif num < 1000:
            return 'Medium (500-999)'
        elif num < 5000:
            return 'Large (1000-4999)'
        else:
            return 'Enterprise (5000+)'
    return 'Unknown'

X_train['OrgSize'] = X_train['OrgSize'].apply(group_org_size)
X_test['OrgSize'] = X_test['OrgSize'].apply(group_org_size)

In [441]:
ohe_org_size = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_train_OrgSize = ohe_org_size.fit_transform(X_train[['OrgSize']])
encoded_test_OrgSize = ohe_org_size.transform(X_test[['OrgSize']])

feature_names = ohe_org_size.get_feature_names_out(['OrgSize'])

train_OrgSize_df = pd.DataFrame(encoded_train_OrgSize, columns=feature_names, index=X_train.index)
test_OrgSize_df = pd.DataFrame(encoded_test_OrgSize, columns=feature_names, index=X_test.index)

X_train = pd.concat([X_train.drop('OrgSize', axis=1), train_OrgSize_df], axis=1)
X_test = pd.concat([X_test.drop('OrgSize', axis=1), test_OrgSize_df], axis=1)

In [442]:
X_train.head(5)

,Age,EdLevel,WorkExp,YearsCode,Country,is_professional,is_employed,DevType_Architecture & Management,DevType_Data & AI,DevType_Design,...,DevType_Other,DevType_Research & Education,DevType_Security,DevType_Software Development,OrgSize_Enterprise (5000+),OrgSize_Freelancer,OrgSize_Large (1000-4999),OrgSize_Medium (500-999),OrgSize_Medium-Small (100-499),OrgSize_Small (20-99)
3822,2.0,4.0,15.0,15.0,Other,1,1,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
12617,2.0,1.0,15.0,17.0,United Kingdom of Great Britain and Northern I...,1,1,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
30992,3.0,6.0,15.0,36.0,Spain,1,1,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
42345,1.0,5.0,6.0,12.0,Other,1,1,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
21712,1.0,5.0,7.0,13.0,Other,1,1,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [443]:
X_train['Country'].value_counts()

Country
Other                                                   4386
United States of America                                4097
Germany                                                 1657
United Kingdom of Great Britain and Northern Ireland    1170
India                                                    847
France                                                   806
Canada                                                   679
Ukraine                                                  546
Netherlands                                              490
Brazil                                                   480
Italy                                                    455
Poland                                                   450
Spain                                                    449
Australia                                                426
Sweden                                                   377
Switzerland                                              329
Czech Republic  

In [444]:
ohe_country = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_train_country = ohe_country.fit_transform(X_train[['Country']])
encoded_test_country = ohe_country.transform(X_test[['Country']])

feature_names = ohe_country.get_feature_names_out(['Country'])

train_country_df = pd.DataFrame(encoded_train_country, columns=feature_names, index=X_train.index)
test_country_df = pd.DataFrame(encoded_test_country, columns=feature_names, index=X_test.index)

X_train = pd.concat([X_train.drop('Country', axis=1), train_country_df], axis=1)
X_test = pd.concat([X_test.drop('Country', axis=1), test_country_df], axis=1)

In [445]:
X_train.head(5)

,Age,EdLevel,WorkExp,YearsCode,is_professional,is_employed,DevType_Architecture & Management,DevType_Data & AI,DevType_Design,DevType_Infrastructure & DevOps,...,Country_Netherlands,Country_Other,Country_Poland,Country_Romania,Country_Spain,Country_Sweden,Country_Switzerland,Country_Ukraine,Country_United Kingdom of Great Britain and Northern Ireland,Country_United States of America
3822,2.0,4.0,15.0,15.0,1,1,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12617,2.0,1.0,15.0,17.0,1,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
30992,3.0,6.0,15.0,36.0,1,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
42345,1.0,5.0,6.0,12.0,1,1,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
21712,1.0,5.0,7.0,13.0,1,1,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Блок 4: Создание нейросетевой модели в PyTorch
Теперь, когда данные готовы, перейдём к созданию нейросети. Определите архитектуру: входной слой (размерность равна количеству признаков после предобработки), несколько скрытых слоёв, выходной слой (1 нейрон для регрессии).

# Блок 5: Обучение модели
Напишите функцию обучения для регрессии. В качестве функции потерь используйте MSELoss (среднеквадратичная ошибка). Можно также считать MAE для оценки. Добавьте раннюю остановку и планировщик learning rate.

# Блок 6: Оценка модели
После обучения оцените модель на тестовой выборке. Поскольку это регрессия, используйте метрики:

MAE (Mean Absolute Error) — средняя абсолютная ошибка в долларах.

RMSE (Root Mean Squared Error) — корень из среднеквадратичной ошибки (чувствителен к выбросам).

R² (коэффициент детерминации) — доля объяснённой дисперсии.

Также постройте график истинных значений против предсказанных.

Ваша задача: Написать код для расчёта метрик и визуализации.

# Блок 7: Эксперименты и улучшения
Попробуйте улучшить модель, экспериментируя с разными аспектами:

Другие наборы признаков.

Разные архитектуры (глубина, ширина, активации).

Другие оптимизаторы (Adam, RMSprop, SGD).

Нормализация целевой переменной (например, логарифмирование зарплаты).

Борьба с выбросами (удаление зарплат выше 99-го процентиля).

Задание: Опишите, какие изменения вы внесли и как они повлияли на метрики. Приложите код и результаты.